# Gains & Guide — Colab에서 LoRA 파인튜닝 (LLaMA-Factory)

**Cursor/AI는 Colab 계정에 직접 연결할 수 없습니다.** 이 노트북을 [Google Colab](https://colab.research.google.com)에서 **파일 → 노트북 업로드**로 열고, GPU 런타임을 선택한 뒤 위에서부터 실행하세요.

**사전 준비**
1. 로컬에서 `cd backend_ai && python -m finetune.build_sft_dataset --validate` 로 `finetune/output/gains_coach_sft_sharegpt.json` 생성
2. [Hugging Face](https://huggingface.co/settings/tokens) 토큰 (Llama 3.x 사용 시 메타 라이선스 동의 필요)
3. 아래 셀에서 데이터 JSON 업로드 또는 Drive 경로 지정

자세한 설명: 레포 `docs/ollama_finetune.md`

In [ ]:
!nvidia-smi

In [ ]:
# LLaMA-Factory 설치 (Colab: 수 분 소요)
%cd /content
!rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -q -e ".[torch,metrics]"

In [ ]:
# Hugging Face 로그인 (Llama 가중치 다운로드용)
from huggingface_hub import login
import os

token = os.environ.get("HF_TOKEN", "")
if not token:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = input("HF_TOKEN 붙여넣기: ").strip()
login(token=token, add_to_git_credential=False)

In [ ]:
# 학습 데이터 넣기 — 방법 A: 업로드
from google.colab import files
import json
import os

DATA_DIR = "/content/LLaMA-Factory/data"
os.makedirs(DATA_DIR, exist_ok=True)

print("gains_coach_sft_sharegpt.json 파일을 선택하세요 (로컬에서 build_sft_dataset 로 생성).")
uploaded = files.upload()
for name in uploaded:
    if name.endswith(".json"):
        open(os.path.join(DATA_DIR, "gains_coach_sft_sharegpt.json"), "wb").write(uploaded[name])
        print("saved:", os.path.join(DATA_DIR, "gains_coach_sft_sharegpt.json"))
        break
else:
    raise SystemExit("JSON 파일을 업로드하지 못했습니다.")

In [ ]:
# dataset_info.json 에 gains_coach_sft 등록 (기존 파일과 병합)
import json
import os

info_path = "/content/LLaMA-Factory/data/dataset_info.json"
entry = {
    "gains_coach_sft": {
        "file_name": "gains_coach_sft_sharegpt.json",
        "formatting": "sharegpt",
        "columns": {"messages": "conversations"},
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_tag": "human",
            "assistant_tag": "gpt",
            "system_tag": "system",
        },
    }
}

if os.path.isfile(info_path):
    with open(info_path, "r", encoding="utf-8") as f:
        base = json.load(f)
else:
    base = {}
base.update(entry)
with open(info_path, "w", encoding="utf-8") as f:
    json.dump(base, f, ensure_ascii=False, indent=2)
print("merged dataset_info.json")

In [ ]:
# Colab용 학습 YAML 작성 (T4: 배치 작게 — VRAM 부족 시 batch 1로 낮추기)
yaml_text = r"""
### model
model_name_or_path: meta-llama/Llama-3.2-3B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_target: all

### dataset
dataset: gains_coach_sft
template: llama3
cutoff_len: 4096
overwrite_cache: true
preprocessing_num_workers: 2

### output
output_dir: saves/gains-coach-lora
logging_steps: 5
save_steps: 50
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
"""
path = "/content/LLaMA-Factory/gains_coach_colab.yaml"
open(path, "w", encoding="utf-8").write(yaml_text.strip() + "\n")
print("wrote", path)

In [ ]:
# 학습 실행 (LLaMA-Factory 버전에 따라 엔트리포인트가 다를 수 있음)
%cd /content/LLaMA-Factory

import os
import shutil
import subprocess

os.chdir("/content/LLaMA-Factory")
cfg = "gains_coach_colab.yaml"
if shutil.which("llamafactory-cli"):
    subprocess.run(["llamafactory-cli", "train", cfg], check=True)
elif shutil.which("llamafactory"):
    subprocess.run(["llamafactory", "train", cfg], check=True)
else:
    subprocess.run(
        ["python", "-m", "llamafactory.cli.train", cfg],
        check=True,
    )

## 다음 단계

- 어댑터 머지 후 GGUF 변환은 로컬 또는 RunPod에서 `llama.cpp`로 진행 (`docs/ollama_finetune.md` 4절).
- Colab 세션 끊기면 `/content`는 사라지므로, `saves/gains-coach-lora` 를 **Drive에 복사**하거나 HF Hub에 `push_to_hub` 하세요.

**CLI 오류 시**: [LLaMA-Factory README](https://github.com/hiyouga/LLaMA-Factory)에서 최신 학습 명령을 확인하세요.